# Data Augmentation Experiment

**Result: no improvement** — Precision@10 unchanged at 0.0556

This notebook documents our attempt to enrich the book metadata and improve the content-based component of our recommendation model. Although the augmentation did not improve the final metric, the experiment provides valuable insights into the limits of metadata enrichment for this dataset.

## Motivation

Our TF-IDF content model relies on book metadata: Title, Author, Subjects, and Publisher. The original dataset has significant gaps:

| Field | Missing |
|---|---|
| Author | 17.4% |
| Subjects | 14.5% |
| Publisher | 0.2% |

The hypothesis: filling in missing fields would give TF-IDF richer signals and improve content-based recommendations.

## Step 1: Data Enrichment Sources

We enriched the original `items.csv` using two external APIs:

**1. Google Books API**
- Queried by ISBN when available
- Filled in missing Author, Subjects, and Publisher fields
- Hit daily quota limits (HTTP 429) — partially enriched ~30% of books

**2. Bibliothèque nationale de France (BnF) API**
- Open API, no quota limits
- Specialised in French books (the majority of this library dataset)
- Filled an additional ~19% of missing Author fields

**Combined result**: ~49% of missing Authors filled. The enriched dataset is saved as `data/augmented/items_enriched.csv`.

## Step 2: Claude AI — Book Classification

In parallel, we used **Claude Haiku** to classify all 15,291 books into 19 thematic categories (law, medicine, fiction, history, etc.).

- Batches of 50 books per API call with prompt caching
- 19 categories covering the full library spectrum
- Result: `data/augmented/books_classified.csv`

The hypothesis: books in the same category are more likely to be co-borrowed, so category similarity could serve as an additional signal.

In [ ]:
import pandas as pd

# Load original vs enriched metadata
items_orig     = pd.read_csv('../data/items.csv')
items_enriched = pd.read_csv('../data/augmented/items_enriched.csv') \
                 if __import__('os').path.exists('../data/augmented/items_enriched.csv') \
                 else items_orig.copy()
classified     = pd.read_csv('../data/augmented/books_classified.csv')

print('=== Original metadata ===')
print((items_orig.isnull().sum() / len(items_orig) * 100).round(1).to_string())
print()
print('=== After enrichment ===')
print((items_enriched.isnull().sum() / len(items_enriched) * 100).round(1).to_string())
print()
print('=== Category distribution (Claude classification) ===')
print(classified['topic_category'].value_counts().head(10).to_string())


## Step 3: 5-Fold CV with Enriched Metadata

We replaced `items.csv` with the enriched version in our best model (Model 6) and ran the same 5-fold cross-validation to measure the impact.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, diags, bmat
import warnings
warnings.filterwarnings('ignore')

def create_weighted_matrix(data, n_users, n_items, decay=0.03):
    mat = np.zeros((n_users, n_items))
    for _, ud in data.groupby('user_id'):
        ud = ud.sort_values('timestamp'); n = len(ud)
        w = np.array([(1-decay)**(n-1-i) for i in range(n)]); w /= w.max()
        for i, (_, row) in enumerate(ud.iterrows()):
            mat[int(row['user_id']), int(row['book_id'])] = w[i]
    return mat

def create_data_matrix(data, n_users, n_items):
    mat = np.zeros((n_users, n_items))
    mat[data['user_id'].values, data['book_id'].values] = 1
    return mat

def ubp(m, s, e=1e-9): return s.dot(m) / (np.abs(s).sum(axis=1)[:,None]+e)
def ibp(m, s, e=1e-9): return (s.dot(m.T)/(s.sum(axis=1)[:,None]+e)).T
def normalize(m): lo,hi=m.min(),m.max(); return (m-lo)/(hi-lo+1e-9)
def sln(x): return normalize(np.log1p(np.abs(x))*np.sign(x))

def precision_recall_at_k(scores, gt, k=10):
    n=scores.shape[0]; tp,tr=0.0,0.0
    for u in range(n):
        ti=np.where(gt[u]==1)[0]; tk=np.argsort(scores[u])[-k:]
        h=np.isin(tk,ti).sum(); tp+=h/k; tr+=h/len(ti) if len(ti)>0 else 0
    return tp/n, tr/n

def build_content(tfidf_mat, train_df, mapping, count_dict, n_users, n_items, book_id_map, books_i):
    profiles=np.zeros((n_users,tfidf_mat.shape[1]),dtype=np.float32)
    ub=train_df.groupby('user_id')['book_id'].apply(list).to_dict()
    for u in range(n_users):
        rows,wts=[],[]
        for b in ub.get(u,[]):
            if b in mapping:
                wts.append(np.log1p(count_dict.get((u,b),1))); rows.append(mapping[b])
        if not rows: continue
        w=np.array(wts,dtype=np.float32); w/=w.sum()
        profiles[u]=np.average(tfidf_mat[rows].toarray(),weights=w,axis=0)
    ns=np.linalg.norm(profiles,axis=1,keepdims=True); ns[ns==0]=1; profiles/=ns
    all_sc=cosine_similarity(profiles,tfidf_mat)
    content=np.zeros((n_users,n_items),dtype=np.float32)
    for i,o in enumerate(books_i):
        if o in book_id_map: content[:,book_id_map[o]]=all_sc[:,i]
    return normalize(np.log1p(content))

def build_graph(bin_mat, n_users, n_items, alpha=0.7, n_iter=15):
    R=csr_matrix(bin_mat)
    adj=bmat([[csr_matrix((n_users,n_users)),R],[R.T,csr_matrix((n_items,n_items))]],format='csr')
    rs=np.array(adj.sum(axis=1)).flatten(); rs[rs==0]=1
    T=diags(1./rs).dot(adj); nt=n_users+n_items
    sc=np.zeros((n_users,n_items))
    for bs in range(0,n_users,200):
        be=min(bs+200,n_users); p=np.zeros((nt,be-bs))
        for i,u in enumerate(range(bs,be)): p[u,i]=1.
        r=p.copy()
        for _ in range(n_iter): r=alpha*T.dot(r)+(1-alpha)*p
        sc[bs:be]=r[n_users:].T
    return normalize(sc)


In [ ]:
# Load data — enriched items
import os
df    = pd.read_csv('../data/interactions_train.csv').rename(
    columns={'u':'user_id','i':'book_id','t':'timestamp'})
books_path = '../data/augmented/items_enriched.csv' \
             if os.path.exists('../data/augmented/items_enriched.csv') \
             else '../data/items.csv'
books = pd.read_csv(books_path)
print(f'Using: {books_path}')

for col in ['Title','Author','Subjects','Publisher']: books[col]=books[col].fillna('')

uid={o:n for n,o in enumerate(df['user_id'].unique())}
bid={o:n for n,o in enumerate(df['book_id'].unique())}
df['user_id']=df['user_id'].map(uid); df['book_id']=df['book_id'].map(bid)
n_users,n_items=df['user_id'].nunique(),df['book_id'].nunique()

books['text']=(books['Title']+' '+books['Author']+' '+books['Author']+' '+
               books['Subjects']+' '+books['Subjects']+' '+books['Publisher'])
tfidf_mat=TfidfVectorizer(max_features=10000,strip_accents='unicode',min_df=2).fit_transform(books['text'])
mapping={bid[o]:i for i,o in enumerate(books['i']) if o in bid}
books_i=books['i'].values

df_s=df.sort_values(['user_id','timestamp']).copy()
df_s['fold']=df_s.groupby('user_id')['timestamp'].transform(
    lambda x: pd.qcut(x.rank(method='first'),5,labels=False))

precisions,recalls=[],[]
for fold in range(5):
    print(f'Fold {fold+1}/5...')
    train=df_s[df_s['fold']!=fold]; test=df_s[df_s['fold']==fold]
    test_mat=np.zeros((n_users,n_items))
    test_mat[test['user_id'].values,test['book_id'].values]=1
    count_dict={(r['user_id'],r['book_id']):r['c']
                for _,r in train.groupby(['user_id','book_id']).size()
                .reset_index(name='c').iterrows()}
    train_mat=create_weighted_matrix(train,n_users,n_items)
    cf=0.45*sln(ubp(train_mat,cosine_similarity(train_mat)))+\
       0.55*sln(ibp(train_mat,cosine_similarity(train_mat.T)))
    content=build_content(tfidf_mat,train,mapping,count_dict,n_users,n_items,bid,books_i)
    bin_mat=create_data_matrix(train,n_users,n_items)
    pop=normalize(np.log1p(bin_mat.sum(axis=0)))
    graph=build_graph(bin_mat,n_users,n_items)
    hybrid=normalize(0.8*normalize(0.75*cf+0.20*content+0.05*pop)+0.2*graph)
    p,r=precision_recall_at_k(hybrid,test_mat)
    precisions.append(p); recalls.append(r)
    print(f'  P@10={p:.4f}  R@10={r:.4f}')

print(f'\nEnriched data → P@10: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}')
print(f'Baseline        → P@10: 0.0556')
print(f'Delta:              {np.mean(precisions)-0.0556:+.4f}')


## Results

### Google Books API Enrichment

We queried the Google Books API using 4 API keys (4,000 requests total). Of the 2,653 books with missing Author:

- 165 had no ISBN → could not be queried
- **716 authors filled** from Google Books API (~27% of missing)
- 1,772 books not found in Google Books (French books with poor coverage)

Missing Author rate: **17.4% → 12.7%**

### 5-Fold CV Comparison

| Model | Precision@10 | Recall@10 |
|---|---|---|
| Model 6 (original `items.csv`) | 0.0556 | 0.2933 |
| Model 6 (enriched, +716 authors) | 0.0556 | 0.2932 |
| **Delta** | **0.0000** | **-0.0001** |

The enriched metadata produced **no measurable improvement** — identical Precision@10 to four decimal places.

## Why Didn't It Help?

After analysis, we identified three likely reasons:

**1. Collaborative filtering dominates**
In our hybrid model, CF carries 75% of the weight. The content component (20%) has limited influence on the final ranking, so improvements to metadata have a small ceiling effect.

**2. TF-IDF already extracts the most useful signal**
The original metadata — even with gaps — already provides enough distinctive vocabulary for TF-IDF to differentiate books. Adding more author names did not introduce qualitatively new signal.

**3. Coverage ceiling for French books**
Google Books has poor coverage of French library books — only 27% of missing authors could be filled despite 4,000 API requests. OpenLibrary returned 0 results for all tested French books.
The BnF (Bibliothèque nationale de France) API, used in an earlier enrichment pass, was the most effective source for this dataset.

## Conclusion

Data augmentation was a thorough and worthwhile experiment, but did not improve the model at any enrichment level tested. The bottleneck is the fundamental sparsity of user-book interactions: with 69% of users having fewer than 10 interactions, collaborative filtering has limited data to work with regardless of how rich the book descriptions are.

> **Key insight:** For sparse interaction datasets, investing in better collaborative filtering algorithms yields more return than enriching content metadata.